## Computes all core OTIF service level metrics defined in the project brief: LIFR, VOFR, OT%, IF%, OTIF% at company, city, customer, and product level. Gap-to-target is computed for every dimension. This notebook is the single source of truth for all headline numbers that feed the Power BI dashboard.

### Analytical Hypotheses

Before computing metrics, eight hypotheses were formed to guide root cause investigation. Each maps to a known supply chain failure mode. Results will be referenced throughout this notebook and the EDA notebook.
##### __H1 -> Forecast/Planning Failure:__ IF failures will cluster on specific SKUs rather than appearing randomly across all products. Under-forecasted SKUs will show disproportionately high IF failure rates.
##### __H2 -> Supplier Shortage:__ IF failures will appear simultaneously across multiple customers ordering the same SKU in the same week, indicating an AtliQ-side supply constraint rather than a customer-specific issue.
##### __H3 -> Warehouse/Fulfilment Error:__ A persistent background rate of small, random IF failures will remain after stripping out SKU-clustered and date-clustered failures, implicating warehouse execution errors.
##### __H4 -> Transportation Issues:__ OT failure rates will be significantly higher in one or more specific cities, pointing to city-level transport as the bottleneck rather than upstream dispatch or planning.
##### __H5 -> Over-Promising:__ Orders with shorter promise windows (agreed delivery date minus order placement date) will show higher OT failure rates, indicating AtliQ commits to lead times it cannot reliably meet.
##### __H6 -> Customer-Side Date Changes:__ Untestable with current data. The dataset captures only the final agreed delivery date, not its history. Recommended for future investigation if AtliQ provides order change logs.
##### __H7 -> Festival Demand Spike:__ IF failure rates will dip sharply in weeks preceding major Indian festivals within the data window (Holi - March, Eid - May, Raksha Bandhan - August), indicating planning does not account for seasonal demand surges.
##### __H8 -> Month-End Capacity Rush:__ OTIF will dip in the last 5 days of each month due to back-loaded shipment volumes as sales teams push to hit monthly quotas.
Hypothesis testing is conducted across 03_core_metrics.ipynb and 04_eda_and_rca.ipynb with findings reported against pre-defined confirmation thresholds.

#### 01. Loading from the /cleaned directory produced by 02_cleaning.ipynb. All corrections and feature engineering are already applied and no transformations needed here.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

CLEAN_DIR = r"E:\Portfolio_Projects\Supply_Chain_FMCG\cleaned"

In [2]:
#Loading cleaned files

master          = pd.read_csv(os.path.join(CLEAN_DIR, 'master.csv'),
                    parse_dates=['order_placement_date',
                                 'agreed_delivery_date',
                                 'actual_delivery_date'])

fact_orders_agg = pd.read_csv(os.path.join(CLEAN_DIR, 'fact_orders_agg.csv'),
                    parse_dates=['order_placement_date'])

dim_targets_orders = pd.read_csv(
                    os.path.join(CLEAN_DIR, 'dim_targets_orders.csv'))

dim_customers   = pd.read_csv(os.path.join(CLEAN_DIR, 'dim_customers.csv'))

print("LOADED SUCCESSFULLY\n")
print(f"  master          : {master.shape}")
print(f"  fact_orders_agg : {fact_orders_agg.shape}")
print(f"  dim_targets     : {dim_targets_orders.shape}")
print(f"  dim_customers   : {dim_customers.shape}")

LOADED SUCCESSFULLY

  master          : (57096, 25)
  fact_orders_agg : (31729, 6)
  dim_targets     : (35, 4)
  dim_customers   : (35, 3)


#### *02. Computing company-level headline metrics across all orders and order lines. These are the numbers that appear on the Executive dashboard: the single most visible figures in the entire project. LIFR and VOFR come from line-level data, OT%, IF%, OTIF% come from order-level data.*

__Company-level headline metrics__

- Line-level metrics (from master table)

In [5]:
total_order_lines = len(master)
lines_in_full     = master['in_full_line'].sum()
total_order_qty   = master['order_qty'].sum()
total_delivered   = master['delivered_qty'].sum()

LIFR  = (lines_in_full / total_order_lines * 100).round(2)
VOFR  = (total_delivered / total_order_qty * 100).round(2)

- Order-level metrics (from fact_orders_agg table)

In [6]:
total_orders  = len(fact_orders_agg)
orders_ot     = fact_orders_agg['on_time'].sum()
orders_if     = fact_orders_agg['in_full'].sum()
orders_otif   = fact_orders_agg['otif'].sum()

OT_pct   = (orders_ot   / total_orders * 100).round(2)
IF_pct   = (orders_if   / total_orders * 100).round(2)
OTIF_pct = (orders_otif / total_orders * 100).round(2)

- Company-level targets (simple average across 35 customers)

In [8]:
avg_ot_target   = dim_targets_orders['ontime_target%'].mean().round(2)
avg_if_target   = dim_targets_orders['infull_target%'].mean().round(2)
avg_otif_target = dim_targets_orders['otif_target%'].mean().round(2)

- Gap to target

In [9]:
ot_gap   = (OT_pct   - avg_ot_target).round(2)
if_gap   = (IF_pct   - avg_if_target).round(2)
otif_gap = (OTIF_pct - avg_otif_target).round(2)

- Summary of all measures

In [10]:
print("COMPANY-LEVEL HEADLINE METRICS\n")
print(f"  {'Metric':<25} {'Actual':>8} {'Target':>8} {'Gap':>8}")
print(f"  {'-'*52}")
print(f"  {'Total Order Lines':<25} {total_order_lines:>8,}")
print(f"  {'Total Orders':<25} {total_orders:>8,}")
print(f"  {'-'*52}")
print(f"  {'LIFR %':<25} {LIFR:>8.2f}")
print(f"  {'VOFR %':<25} {VOFR:>8.2f}")
print(f"  {'-'*52}")
print(f"  {'OT %':<25} {OT_pct:>8.2f} {avg_ot_target:>8.2f} {ot_gap:>8.2f}")
print(f"  {'IF %':<25} {IF_pct:>8.2f} {avg_if_target:>8.2f} {if_gap:>8.2f}")
print(f"  {'OTIF %':<25} {OTIF_pct:>8.2f} {avg_otif_target:>8.2f} {otif_gap:>8.2f}")

COMPANY-LEVEL HEADLINE METRICS

  Metric                      Actual   Target      Gap
  ----------------------------------------------------
  Total Order Lines           57,096
  Total Orders                31,729
  ----------------------------------------------------
  LIFR %                       65.96
  VOFR %                       96.59
  ----------------------------------------------------
  OT %                         59.03    86.09   -27.06
  IF %                         52.78    76.51   -23.73
  OTIF %                       29.02    65.91   -36.89


### OTIF at 29.02% means AtliQ fails to meet the basic customer promise on 7 in every 10 orders; this is a systemic failure, not an isolated one. The 30-point gap between LIFR (65.96%) and VOFR (96.59%) reveals that shortfalls are small in volume but frequent in occurrence. AtliQ's strict binary OTIF criterion amplifies small supply gaps into large metric failures, explaining why customer experience deteriorated faster than volume metrics suggested.

#### *03. Breaking down OT%, IF%, and OTIF% by city to identify whether service failures are concentrated in specific geographies. City-level concentration would implicate transportation or local warehouse issues rather than a company-wide supply problem.*

__City-level metrics__

- Joining city to fact_orders_agg 

In [12]:
orders_with_city = fact_orders_agg.merge(
    dim_customers[['customer_id', 'city']],
    on='customer_id', how='left'
)

- Compute city-level metrics

In [13]:
city_metrics = orders_with_city.groupby('city').agg(
    total_orders  = ('order_id',  'count'),
    orders_ot     = ('on_time',   'sum'),
    orders_if     = ('in_full',   'sum'),
    orders_otif   = ('otif',      'sum')
).reset_index()

city_metrics['OT_%']   = (city_metrics['orders_ot']   / city_metrics['total_orders'] * 100).round(2)
city_metrics['IF_%']   = (city_metrics['orders_if']   / city_metrics['total_orders'] * 100).round(2)
city_metrics['OTIF_%'] = (city_metrics['orders_otif'] / city_metrics['total_orders'] * 100).round(2)

- Adding targets and gaps

In [14]:
city_metrics['OT_target']   = avg_ot_target
city_metrics['IF_target']   = avg_if_target
city_metrics['OTIF_target'] = avg_otif_target

city_metrics['OT_gap']   = (city_metrics['OT_%']   - avg_ot_target).round(2)
city_metrics['IF_gap']   = (city_metrics['IF_%']   - avg_if_target).round(2)
city_metrics['OTIF_gap'] = (city_metrics['OTIF_%'] - avg_otif_target).round(2)

- Summary of above measures

In [15]:
print("CITY-LEVEL METRICS\n")
print(f"  {'City':<12} {'Orders':>8} {'OT%':>7} {'IF%':>7} {'OTIF%':>7} {'OT_gap':>8} {'IF_gap':>8} {'OTIF_gap':>10}")
print(f"  {'-'*72}")
for _, row in city_metrics.iterrows():
    print(f"  {row['city']:<12} {row['total_orders']:>8,.0f} {row['OT_%']:>7.2f} {row['IF_%']:>7.2f} {row['OTIF_%']:>7.2f} {row['OT_gap']:>8.2f} {row['IF_gap']:>8.2f} {row['OTIF_gap']:>10.2f}")

CITY-LEVEL METRICS

  City           Orders     OT%     IF%   OTIF%   OT_gap   IF_gap   OTIF_gap
  ------------------------------------------------------------------------
  Ahmedabad      11,061   58.16   54.20   29.33   -27.93   -22.31     -36.58
  Surat           9,696   61.21   52.55   30.07   -24.88   -23.96     -35.84
  Vadodara       10,972   57.98   51.56   27.78   -28.11   -24.95     -38.13


### City-level OTIF gaps are consistent across all three cities i.e., Ahmedabad -36.58, Surat -35.84, Vadodara -38.13. No single city shows dramatically worse performance. This rules out city-specific transportation as the primary failure driver (H4). The near-uniform failure rate across geographies points to an upstream cause affecting all cities equally; likely at planning, supply, or dispatch level.

#### *04. Computing OT%, IF%, and OTIF% at individual customer level and measuring gap against each customer's own target. Unlike city-level analysis which uses average targets, here each customer is measured against their specific contracted target. This is the most granular and actionable service level view.*

__Customer-level metrics__

- Joining customer info and targets to fact_orders_agg

In [16]:
orders_with_customer = fact_orders_agg.merge(
    dim_customers, on='customer_id', how='left'
).merge(
    dim_targets_orders, on='customer_id', how='left'
)

- Computing customer-level metrics

In [17]:
customer_metrics = orders_with_customer.groupby(
    ['customer_id', 'customer_name', 'city']
).agg(
    total_orders  = ('order_id', 'count'),
    orders_ot     = ('on_time',  'sum'),
    orders_if     = ('in_full',  'sum'),
    orders_otif   = ('otif',     'sum'),
    ot_target     = ('ontime_target%', 'first'),
    if_target     = ('infull_target%', 'first'),
    otif_target   = ('otif_target%',   'first')
).reset_index()

customer_metrics['OT_%']   = (customer_metrics['orders_ot']   / customer_metrics['total_orders'] * 100).round(2)
customer_metrics['IF_%']   = (customer_metrics['orders_if']   / customer_metrics['total_orders'] * 100).round(2)
customer_metrics['OTIF_%'] = (customer_metrics['orders_otif'] / customer_metrics['total_orders'] * 100).round(2)

- Gap against individual customer targets

In [18]:
customer_metrics['OT_gap']   = (customer_metrics['OT_%']   - customer_metrics['ot_target']).round(2)
customer_metrics['IF_gap']   = (customer_metrics['IF_%']   - customer_metrics['if_target']).round(2)
customer_metrics['OTIF_gap'] = (customer_metrics['OTIF_%'] - customer_metrics['otif_target']).round(2)

- Summarizing sorted by OTIF gap ascending (worst first)

In [19]:
customer_display = customer_metrics[[
    'customer_name', 'city', 'total_orders',
    'OT_%', 'IF_%', 'OTIF_%',
    'ot_target', 'if_target', 'otif_target',
    'OT_gap', 'IF_gap', 'OTIF_gap'
]].sort_values('OTIF_gap', ascending=True)

print("CUSTOMER-LEVEL METRICS - SORTED BY OTIF GAP (WORST FIRST)\n")
print(customer_display.to_string(index=False))

CUSTOMER-LEVEL METRICS - SORTED BY OTIF GAP (WORST FIRST)

    customer_name      city  total_orders  OT_%  IF_%  OTIF_%  ot_target  if_target  otif_target  OT_gap  IF_gap  OTIF_gap
      Info Stores     Surat           827 70.74 19.35    9.43         92         67           62  -21.26  -47.65    -52.57
     Vijay Stores  Vadodara           815 74.97 17.91   10.55         92         67           62  -17.03  -49.09    -51.45
       Elite Mart  Vadodara           813 72.20 16.61    9.72         92         65           60  -19.80  -48.39    -50.28
     Sorefoz Mart Ahmedabad           832 71.63 17.67   10.70         89         66           59  -17.37  -48.33    -48.30
       Lotus Mart  Vadodara          1168 27.83 67.29   19.69         79         81           64  -51.17  -13.71    -44.31
 Acclaimed Stores     Surat          1126 29.84 22.38    6.93         75         68           51  -45.16  -45.62    -44.07
       Lotus Mart Ahmedabad          1179 28.33 23.83    7.97         78        

### Customer-level analysis reveals two structurally different failure profiles. The first group, including Info Stores, Vijay Stores, Elite Mart, and Sorefoz Mart, shows OT% around 70-75% but IF% collapsing to 16-19%, indicating a severe supply availability problem specific to these customers. The second group, including Lotus Mart, Acclaimed Stores, and Coolblue, shows IF% around 66-68% but OT% collapsing to 27-29%, indicating a chronic delivery timing problem. These two profiles require entirely different operational interventions and should not be treated as a single uniform service failure.

#### *05. Computing LIFR and VOFR at product and category level. As per the stakeholder brief, only line-level fill metrics are reported at product level - OT is not measurable at product level since on-time is an order-level concept. Category-level breakdown tests whether Dairy, Food, and Beverages differ meaningfully in supply reliability.*

__Product and category level metrics__

- Category-level LIFR and VOFR

In [20]:
category_metrics = master.groupby('category').agg(
    total_lines    = ('in_full_line', 'count'),
    lines_in_full  = ('in_full_line', 'sum'),
    total_ordered  = ('order_qty',    'sum'),
    total_delivered= ('delivered_qty','sum')
).reset_index()

category_metrics['LIFR_%'] = (category_metrics['lines_in_full'] / category_metrics['total_lines']   * 100).round(2)
category_metrics['VOFR_%'] = (category_metrics['total_delivered']/ category_metrics['total_ordered'] * 100).round(2)

print("CATEGORY-LEVEL METRICS\n")
print(f"  {'Category':<12} {'Lines':>8} {'LIFR%':>8} {'VOFR%':>8}")
print(f"  {'-'*40}")
for _, row in category_metrics.iterrows():
    print(f"  {row['category']:<12} {row['total_lines']:>8,} {row['LIFR_%']:>8.2f} {row['VOFR_%']:>8.2f}")

CATEGORY-LEVEL METRICS

  Category        Lines    LIFR%    VOFR%
  ----------------------------------------
  Beverages       9,461    65.54    96.54
  Dairy          38,096    65.95    96.59
  Food            9,539    66.43    96.64


- Product-level LIFR and VOFR

In [21]:
product_metrics = master.groupby(
    ['product_id', 'product_name', 'category']
).agg(
    total_lines     = ('in_full_line', 'count'),
    lines_in_full   = ('in_full_line', 'sum'),
    total_ordered   = ('order_qty',    'sum'),
    total_delivered = ('delivered_qty','sum')
).reset_index()

product_metrics['LIFR_%'] = (product_metrics['lines_in_full'] / product_metrics['total_lines']    * 100).round(2)
product_metrics['VOFR_%'] = (product_metrics['total_delivered']/ product_metrics['total_ordered'] * 100).round(2)
product_metrics['IF_failure_rate'] = (100 - product_metrics['LIFR_%']).round(2)

product_display = product_metrics[[
    'product_name', 'category',
    'total_lines', 'LIFR_%', 'VOFR_%', 'IF_failure_rate'
]].sort_values('IF_failure_rate', ascending=False)

print("\nPRODUCT-LEVEL METRICS - SORTED BY IF FAILURE RATE (WORST FIRST)\n")
print(product_display.to_string(index=False))


PRODUCT-LEVEL METRICS - SORTED BY IF FAILURE RATE (WORST FIRST)

   product_name  category  total_lines  LIFR_%  VOFR_%  IF_failure_rate
  AM Butter 250     Dairy         3125   63.52   96.36            36.48
     AM Tea 250 Beverages         3143   65.16   96.52            34.84
AM Biscuits 250      Food         3186   65.16   96.58            34.84
  AM Butter 500     Dairy         3272   65.19   96.46            34.81
    AM Ghee 250     Dairy         3200   65.25   96.53            34.75
     AM Tea 100 Beverages         3134   65.32   96.59            34.68
     AM Curd 50     Dairy         3187   65.55   96.62            34.45
    AM Milk 100     Dairy         3184   65.55   96.54            34.45
    AM Ghee 100     Dairy         3098   65.75   96.59            34.25
    AM Milk 250     Dairy         3197   65.91   96.61            34.09
AM Biscuits 500      Food         3195   66.10   96.49            33.90
     AM Tea 500 Beverages         3184   66.14   96.52            33.8

### IF failure rates across all 18 products range from 31.95% to 36.48%, a spread of only 4.53 percentage points. This near-uniform distribution across products effectively challenges H1 - if forecast or planning failure were the primary driver, we would expect sharp concentration in specific SKUs. Instead, every product is failing at similar rates, pointing to a systemic supply or capacity constraint that affects the entire portfolio rather than individual product forecast errors. H1 weakly supported - SKU-level concentration is absent, upstream capacity constraint is implicated.

#### *06. Analyzing OTIF performance across months, weeks, and day-of-month buckets. Monthly trends reveal overall trajectory. Weekly patterns surface operational rhythm issues. Day-of-month bucketing tests H8 - whether month-end shipment back-loading creates a capacity crunch that depresses OTIF in the final days of each month.*

__Time-based metrics__

- Adding time columns to fact_orders_agg

In [23]:
fact_orders_agg['order_month']  = fact_orders_agg['order_placement_date'].dt.to_period('M')
fact_orders_agg['order_week']   = fact_orders_agg['order_placement_date'].dt.isocalendar().week
fact_orders_agg['day_of_month'] = fact_orders_agg['order_placement_date'].dt.day

- Monthly OTIF trend

In [24]:
monthly = fact_orders_agg.groupby('order_month').agg(
    total_orders = ('order_id', 'count'),
    orders_ot    = ('on_time',  'sum'),
    orders_if    = ('in_full',  'sum'),
    orders_otif  = ('otif',     'sum')
).reset_index()

monthly['OT_%']   = (monthly['orders_ot']   / monthly['total_orders'] * 100).round(2)
monthly['IF_%']   = (monthly['orders_if']   / monthly['total_orders'] * 100).round(2)
monthly['OTIF_%'] = (monthly['orders_otif'] / monthly['total_orders'] * 100).round(2)

print("MONTHLY OTIF TREND\n")
print(f"  {'Month':<10} {'Orders':>8} {'OT%':>7} {'IF%':>7} {'OTIF%':>7}")
print(f"  {'-'*45}")
for _, row in monthly.iterrows():
    print(f"  {str(row['order_month']):<10} {row['total_orders']:>8,} {row['OT_%']:>7.2f} {row['IF_%']:>7.2f} {row['OTIF_%']:>7.2f}")

MONTHLY OTIF TREND

  Month        Orders     OT%     IF%   OTIF%
  ---------------------------------------------
  2022-03       5,407   59.57   52.34   28.87
  2022-04       5,253   59.32   52.56   28.67
  2022-05       5,417   58.50   53.66   29.13
  2022-06       5,213   58.51   52.04   28.72
  2022-07       5,339   59.39   52.48   29.35
  2022-08       5,100   58.88   53.61   29.39


- Day-of-month bucket analysis (H8)

In [25]:
def day_bucket(d):
    if d <= 10:
        return '01-10'
    elif d <= 20:
        return '11-20'
    else:
        return '21-end'

fact_orders_agg['day_bucket'] = fact_orders_agg['day_of_month'].apply(day_bucket)

dom_metrics = fact_orders_agg.groupby('day_bucket').agg(
    total_orders = ('order_id', 'count'),
    orders_ot    = ('on_time',  'sum'),
    orders_if    = ('in_full',  'sum'),
    orders_otif  = ('otif',     'sum')
).reset_index()

dom_metrics['OT_%']   = (dom_metrics['orders_ot']   / dom_metrics['total_orders'] * 100).round(2)
dom_metrics['IF_%']   = (dom_metrics['orders_if']   / dom_metrics['total_orders'] * 100).round(2)
dom_metrics['OTIF_%'] = (dom_metrics['orders_otif'] / dom_metrics['total_orders'] * 100).round(2)

print("\nDAY-OF-MONTH BUCKET ANALYSIS - H8 TEST\n")
print(f"  {'Day Bucket':<10} {'Orders':>8} {'OT%':>7} {'IF%':>7} {'OTIF%':>7}")
print(f"  {'-'*45}")
for _, row in dom_metrics.sort_values('day_bucket').iterrows():
    print(f"  {row['day_bucket']:<10} {row['total_orders']:>8,} {row['OT_%']:>7.2f} {row['IF_%']:>7.2f} {row['OTIF_%']:>7.2f}")


DAY-OF-MONTH BUCKET ANALYSIS - H8 TEST

  Day Bucket   Orders     OT%     IF%   OTIF%
  ---------------------------------------------
  01-10        10,445   58.90   52.49   28.95
  11-20        10,503   59.22   52.62   29.12
  21-end       10,781   58.97   53.21   29.00


### OTIF remained structurally flat across all 6 months at approximately 29%, with a total range of 0.72 percentage points. This flat trend indicates AtliQ has normalized failure rather than experiencing a temporary disruption. No seasonal or festival-driven pattern is detectable within the March to August 2022 window, weakening H7. Day-of-month bucket analysis shows negligible variation across early, mid, and late month periods with a maximum spread of 0.17 percentage points, effectively rejecting H8. The absence of any time-based pattern reinforces the conclusion that the failure driver is structural and upstream rather than seasonal or rhythmic.

#### *07. Saving all computed metric tables to the /outputs folder for reference in downstream notebooks and for import into the SQL layer. These CSVs represent the complete analytical output of the core metrics notebook.*

__Saving metric outputs__

In [26]:
OUTPUT_DIR = r"E:\Portfolio_Projects\Supply_Chain_FMCG\outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

city_metrics.to_csv(
    os.path.join(OUTPUT_DIR, 'city_metrics.csv'), index=False)
customer_metrics.to_csv(
    os.path.join(OUTPUT_DIR, 'customer_metrics.csv'), index=False)
product_metrics.to_csv(
    os.path.join(OUTPUT_DIR, 'product_metrics.csv'), index=False)
category_metrics.to_csv(
    os.path.join(OUTPUT_DIR, 'category_metrics.csv'), index=False)
monthly.to_csv(
    os.path.join(OUTPUT_DIR, 'monthly_metrics.csv'), index=False)
dom_metrics.to_csv(
    os.path.join(OUTPUT_DIR, 'dom_metrics.csv'), index=False)

print("METRIC OUTPUTS SAVED\n")
for f in sorted(os.listdir(OUTPUT_DIR)):
    filepath = os.path.join(OUTPUT_DIR, f)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  {f:40s}  {size_kb:>6.1f} KB")

METRIC OUTPUTS SAVED

  category_metrics.csv                         0.2 KB
  city_metrics.csv                             0.4 KB
  customer_metrics.csv                         3.3 KB
  dom_metrics.csv                              0.2 KB
  monthly_metrics.csv                          0.3 KB
  product_metrics.csv                          1.4 KB
